# Stacking, Calibration, and Label-Blind Prediction

This notebook documents the revised hybrid-stacking stage. It consumes the repeated-seed base-model probabilities, trains XGBoost stacking and validation-based calibration, and creates label-blind locked-test predictions for later repeated-seed evaluation.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level predictions, checkpoints, fold models, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the stacking and calibration protocol

Imports the modelling and evaluation libraries, defines the restricted project-root placeholder, declares upstream and downstream artifact locations, fixes planned seeds, and specifies the five final models, XGBoost recipe, calibration recipe, and stacking meta-features.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_06_ROOT = ROUND1_DIR / 'artifacts' / 'r1_06_repeated_seed_base_oof_v2'
R1_07_ROOT = ROUND1_DIR / 'artifacts' / 'r1_07_revised_stacking_calibration_v1'

VALIDATION_SOURCE_PATH = PROJECT_ROOT / 'data' / 'splits' / 'validation.csv'

PLANNED_SEEDS = [42, 52, 62]
RUN_SEEDS = [62]
MAX_NEW_SEEDS_PER_RUN: int | None = None

LABEL_NAMES = ['negatif', 'netral', 'positif']
LABEL_TO_ID = {label: index for index, label in enumerate(LABEL_NAMES)}
N_CLASSES = len(LABEL_NAMES)
GROUP_COLUMN = 'Actual Place Name'
LABEL_COLUMN = 'sentiment_label'
META_FEATURE_NAMES = [
    'svm_prob_negatif', 'svm_prob_netral', 'svm_prob_positif',
    'bert_prob_negatif', 'bert_prob_netral', 'bert_prob_positif',
    'svm_confidence', 'bert_confidence', 'svm_entropy', 'bert_entropy',
    'pred_disagreement', 'prob_disagreement',
]
MODEL_ORDER = [
    ('svm', 'SVM'),
    ('indobert', 'IndoBERT'),
    ('wa_svm_bert', 'WA (SVM+BERT)'),
    ('xgboost', 'XGBoost'),
    ('calibrated_xgboost', 'Calibrated XGBoost'),
]
XGB_CONFIG = {
    'objective': 'multi:softprob', 'num_class': N_CLASSES,
    'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8,
    'eval_metric': 'mlogloss', 'n_jobs': -1, 'verbosity': 0,
}
CALIBRATION_CONFIG = {
    'method': 'multiclass_platt_logistic_regression_on_xgboost_probabilities',
    'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000,
    'crossfit_n_splits': 5, 'crossfit_group_aware': True,
}
PROTOCOL_VERSION = 'r1_07_protocol_v1_oof_stacking_validation_platt'

print(f'Restricted project root placeholder: {PROJECT_ROOT}')
print(f'Base-model input: {R1_06_ROOT}')
print(f'Stacking and calibration output: {R1_07_ROOT}')
print(f'Run seeds: {RUN_SEEDS} of planned {PLANNED_SEEDS}')
print(f'XGBoost: {xgboost.__version__} | sklearn: {sklearn.__version__}')


### Define shared persistence and probability utilities

Defines timestamp, checksum, atomic-write, probability-validation, entropy, accuracy, macro-F1, ECE, Brier-score, and seed-directory helpers used by the stacking workflow.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json_hash(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()


def derive_seed(base_seed: int, purpose: str) -> int:
    raw = hashlib.sha256(f'{base_seed}:{purpose}'.encode('utf-8')).digest()
    return int.from_bytes(raw[:8], byteorder='little') % (2**31 - 1)


def atomic_write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    temporary.replace(path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def atomic_dump_joblib(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp')
    joblib.dump(value, temporary)
    temporary.replace(path)


def probability_columns(model_key: str) -> list[str]:
    return [f'{model_key}_prob_{label}' for label in LABEL_NAMES]


def assert_exact_row_ids(frame: pd.DataFrame, expected_n: int, name: str) -> None:
    actual = frame['row_id'].to_numpy(dtype=np.int64)
    expected = np.arange(expected_n, dtype=np.int64)
    if len(actual) != expected_n or not np.array_equal(actual, expected):
        raise ValueError(f'{name}: row_id harus berurutan dari 0 sampai {expected_n - 1}.')


def validate_probability_matrix(probabilities: np.ndarray, name: str) -> None:
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != N_CLASSES:
        raise ValueError(f'{name}: bentuk probability tidak valid: {probabilities.shape}')
    if not np.isfinite(probabilities).all():
        raise ValueError(f'{name}: terdapat NaN atau infinity.')
    if np.any(probabilities < -1e-10) or np.any(probabilities > 1 + 1e-10):
        raise ValueError(f'{name}: probability di luar [0, 1].')
    maximum_error = float(np.max(np.abs(probabilities.sum(axis=1) - 1.0)))
    if maximum_error > 1e-5:
        raise ValueError(f'{name}: row probability sum tidak satu. Error={maximum_error:.3e}')


def extract_probabilities(frame: pd.DataFrame, model_key: str, name: str) -> np.ndarray:
    columns = probability_columns(model_key)
    if set(columns) - set(frame.columns):
        raise ValueError(f'{name}: kolom probability {model_key} tidak lengkap.')
    probabilities = frame[columns].to_numpy(dtype=np.float64)
    validate_probability_matrix(probabilities, f'{name}/{model_key}')
    return probabilities


def entropy(probabilities: np.ndarray) -> np.ndarray:
    safe = np.clip(probabilities, 1e-12, 1.0)
    return -(safe * np.log(safe)).sum(axis=1)


def compute_ece(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 10) -> float:
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    class_scores = []
    for class_index in range(N_CLASSES):
        probability = probabilities[:, class_index]
        actual = (y_true == class_index).astype(float)
        score = 0.0
        for left, right in zip(edges[:-1], edges[1:]):
            mask = (probability >= left) & ((probability <= right) if right == 1.0 else (probability < right))
            if mask.any():
                score += float(mask.mean() * abs(actual[mask].mean() - probability[mask].mean()))
        class_scores.append(score)
    return float(np.mean(class_scores))


def compute_brier(probabilities: np.ndarray, y_true: np.ndarray) -> float:
    return float(np.mean([
        brier_score_loss((y_true == class_index).astype(float), probabilities[:, class_index])
        for class_index in range(N_CLASSES)
    ]))


def metric_summary(y_true: np.ndarray, probabilities: np.ndarray) -> dict:
    validate_probability_matrix(probabilities, 'metric probabilities')
    prediction = probabilities.argmax(axis=1)
    return {
        'n': int(len(y_true)),
        'accuracy': float(accuracy_score(y_true, prediction)),
        'macro_f1': float(f1_score(y_true, prediction, average='macro', zero_division=0)),
        'ece': compute_ece(probabilities, y_true),
        'brier_score': compute_brier(probabilities, y_true),
    }


def seed_input_directory(seed: int) -> Path:
    return R1_06_ROOT / f'seed_{seed}'


def seed_output_directory(seed: int) -> Path:
    return R1_07_ROOT / f'seed_{seed}'


### Load and validate base-model seed artifacts

Reads the authorized base-model OOF, validation, and label-blind locked-test probabilities for one seed. It verifies row order, probability structure, the five-fold ensemble contract, and the absence of labels in the locked-test artifact.


In [ ]:
def load_r1_06_seed(seed: int) -> dict:
    input_dir = seed_input_directory(seed)
    paths = {
        'run_manifest': input_dir / 'run_manifest.json',
        'run_state': input_dir / 'run_state.json',
        'oof_base': input_dir / 'oof_train_base_probabilities.csv',
        'validation_base': input_dir / 'validation_ensemble_base_probabilities.csv',
        'locked_base': input_dir / 'locked_test_ensemble_base_probabilities_blind.csv',
        'oof_meta': input_dir / 'oof_train_meta_features.csv',
        'validation_meta': input_dir / 'validation_meta_features.csv',
        'locked_meta': input_dir / 'locked_test_meta_features_blind.csv',
    }
    missing = [name for name, path in paths.items() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Base-model seed {seed} belum lengkap: {missing}')

    manifest = json.loads(paths['run_manifest'].read_text(encoding='utf-8'))
    state = json.loads(paths['run_state'].read_text(encoding='utf-8'))
    if manifest.get('seed') != seed or state.get('status') != 'seed_completed':
        raise RuntimeError(f'Base-model seed {seed} belum completed.')
    if manifest.get('locked_test_metrics_computed') is not False or manifest.get('locked_test_artifact_is_label_blind') is not True:
        raise RuntimeError('Base-model locked-test guardrail tidak terpenuhi.')

    protocol_path = R1_06_ROOT / 'protocol_v2.json'
    protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
    if protocol.get('scientific_protocol', {}).get('protocol_version') != 'r1_06_protocol_v2_outer_train_refit':
        raise RuntimeError('Stacking and calibration accepts only the base-model outer-train refit protocol.')

    frames = {name: pd.read_csv(path) for name, path in paths.items() if name not in {'run_manifest', 'run_state'}}
    oof_base, validation_base, locked_base = frames['oof_base'], frames['validation_base'], frames['locked_base']
    oof_meta, validation_meta, locked_meta = frames['oof_meta'], frames['validation_meta'], frames['locked_meta']

    for name, frame in [('OOF', oof_base), ('validation', validation_base), ('locked test', locked_base),
                        ('OOF meta', oof_meta), ('validation meta', validation_meta), ('locked meta', locked_meta)]:
        assert_exact_row_ids(frame, len(frame), name)
    if not np.array_equal(oof_base['row_id'], oof_meta['row_id']):
        raise RuntimeError('Urutan OOF base dan meta berbeda.')
    if not np.array_equal(validation_base['row_id'], validation_meta['row_id']):
        raise RuntimeError('Urutan validation base dan meta berbeda.')
    if not np.array_equal(locked_base['row_id'], locked_meta['row_id']):
        raise RuntimeError('Urutan locked-test base dan meta berbeda.')

    if 'y_silver' not in oof_base or 'y_silver' not in validation_base:
        raise RuntimeError('OOF dan validation harus memiliki y_silver.')
    if 'y_silver' in locked_base or 'y_silver' in locked_meta:
        raise RuntimeError('Locked-test input tidak lagi blind.')
    if list(oof_meta.columns) != ['row_id', 'outer_fold', 'y_silver', *META_FEATURE_NAMES]:
        raise RuntimeError('Skema OOF meta-feature berubah.')
    if list(validation_meta.columns) != ['row_id', 'y_silver', *META_FEATURE_NAMES]:
        raise RuntimeError('Skema validation meta-feature berubah.')
    if list(locked_meta.columns) != ['row_id', *META_FEATURE_NAMES]:
        raise RuntimeError('Skema locked-test meta-feature berubah.')

    for name, frame in [('OOF', oof_base), ('validation', validation_base), ('locked test', locked_base)]:
        extract_probabilities(frame, 'svm', name)
        extract_probabilities(frame, 'bert', name)

    validation_source = pd.read_csv(VALIDATION_SOURCE_PATH)
    if len(validation_source) != len(validation_base):
        raise RuntimeError('Jumlah validation.csv tidak cocok dengan base-model output.')
    source_y = validation_source[LABEL_COLUMN].map(LABEL_TO_ID).to_numpy(dtype=np.int64)
    if not np.array_equal(source_y, validation_base['y_silver'].to_numpy(dtype=np.int64)):
        raise RuntimeError('Label atau urutan validation source berbeda dari base-model output.')
    if GROUP_COLUMN not in validation_source:
        raise RuntimeError(f'validation.csv kehilangan kolom group {GROUP_COLUMN}.')

    input_hashes = {name: sha256_file(path) for name, path in paths.items()}
    input_hashes['r1_06_protocol_v2_sha256'] = sha256_file(protocol_path)
    print(f'Base-model seed {seed}: OOF={len(oof_base):,}, validation={len(validation_base):,}, locked blind={len(locked_base):,}')
    return {
        'seed': seed, 'input_dir': input_dir, 'run_manifest': manifest, 'protocol': protocol,
        'input_hashes': input_hashes, 'oof_base': oof_base, 'validation_base': validation_base,
        'locked_base': locked_base, 'oof_meta': oof_meta, 'validation_meta': validation_meta,
        'locked_meta': locked_meta, 'validation_groups': validation_source[GROUP_COLUMN].fillna('').astype(str).to_numpy(),
    }


### Define final-model and metric builders

Defines XGBoost stacking, validation-calibration, weighted-average fusion, metric calculations, and functions that assemble the five pre-specified final-model prediction outputs.


In [ ]:
def build_xgboost(seed: int) -> XGBClassifier:
    return XGBClassifier(**XGB_CONFIG, random_state=derive_seed(seed, 'xgboost_12_feature'))


def build_platt(seed: int, purpose: str) -> LogisticRegression:
    return LogisticRegression(
        C=CALIBRATION_CONFIG['C'],
        max_iter=CALIBRATION_CONFIG['max_iter'],
        solver=CALIBRATION_CONFIG['solver'],
        random_state=derive_seed(seed, purpose),
    )


def weighted_average(svm_probabilities: np.ndarray, bert_probabilities: np.ndarray, weights: dict) -> np.ndarray:
    combined = weights['svm'] * svm_probabilities + weights['indobert'] * bert_probabilities
    combined /= combined.sum(axis=1, keepdims=True)
    validate_probability_matrix(combined, 'weighted average')
    return combined


def derive_wa_weights(y_oof: np.ndarray, svm_oof: np.ndarray, bert_oof: np.ndarray) -> dict:
    svm_f1 = float(f1_score(y_oof, svm_oof.argmax(axis=1), average='macro', zero_division=0))
    bert_f1 = float(f1_score(y_oof, bert_oof.argmax(axis=1), average='macro', zero_division=0))
    total = svm_f1 + bert_f1
    if total <= 0:
        raise RuntimeError('OOF macro-F1 tidak valid untuk bobot WA.')
    return {
        'svm': svm_f1 / total, 'indobert': bert_f1 / total, 'source': 'OOF macro-F1 only',
        'svm_oof_macro_f1': svm_f1, 'indobert_oof_macro_f1': bert_f1,
    }


def model_prediction_frame(
    row_ids: np.ndarray,
    model_probabilities: dict[str, np.ndarray],
    y_silver: np.ndarray | None,
    model_order: list[tuple[str, str]] = MODEL_ORDER,
) -> pd.DataFrame:
    frame = pd.DataFrame({'row_id': row_ids.astype(np.int64)})
    if y_silver is not None:
        frame['y_silver'] = y_silver.astype(np.int64)
    for model_key, _ in model_order:
        probabilities = model_probabilities[model_key]
        validate_probability_matrix(probabilities, f'{model_key} output')
        for class_index, class_name in enumerate(LABEL_NAMES):
            frame[f'{model_key}_prob_{class_name}'] = probabilities[:, class_index]
        frame[f'{model_key}_prediction'] = probabilities.argmax(axis=1).astype(np.int64)
        frame[f'{model_key}_confidence'] = probabilities.max(axis=1)
        frame[f'{model_key}_entropy'] = entropy(probabilities)
    return frame


def feature_importance_frame(xgb_model: XGBClassifier) -> pd.DataFrame:
    importance = np.asarray(xgb_model.feature_importances_, dtype=np.float64)
    if len(importance) != len(META_FEATURE_NAMES):
        raise RuntimeError('Feature importance tidak cocok dengan schema.')
    result = pd.DataFrame({'feature': META_FEATURE_NAMES, 'importance_gain': importance})
    result = result.sort_values('importance_gain', ascending=False, ignore_index=True)
    result.insert(0, 'rank', np.arange(1, len(result) + 1))
    return result


### Cross-fit probability calibration on validation data

Uses group-aware folds inside validation data to obtain cross-fitted Platt calibration probabilities, then fits one complete validation calibrator for label-blind locked-test deployment.


In [ ]:
def fit_cross_fitted_platt(
    xgb_validation_probabilities: np.ndarray,
    y_validation: np.ndarray,
    validation_groups: np.ndarray,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, LogisticRegression, list[dict]]:
    splitter = StratifiedGroupKFold(
        n_splits=CALIBRATION_CONFIG['crossfit_n_splits'],
        shuffle=True,
        random_state=derive_seed(seed, 'calibration_crossfit_split'),
    )
    calibrated = np.full_like(xgb_validation_probabilities, np.nan, dtype=np.float64)
    assignments = np.full(len(y_validation), -1, dtype=np.int64)
    records = []

    for fold, (fit_idx, eval_idx) in enumerate(splitter.split(xgb_validation_probabilities, y_validation, validation_groups), start=1):
        fit_groups, eval_groups = set(validation_groups[fit_idx]), set(validation_groups[eval_idx])
        overlap = fit_groups.intersection(eval_groups)
        if overlap:
            raise RuntimeError(f'Calibration fold {fold} memiliki group overlap.')
        calibrator = build_platt(seed, f'platt_calibration_fold_{fold}')
        calibrator.fit(xgb_validation_probabilities[fit_idx], y_validation[fit_idx])
        if not np.array_equal(calibrator.classes_, np.arange(N_CLASSES)):
            raise RuntimeError(f'Calibration fold {fold} tidak memiliki tiga kelas lengkap.')
        calibrated[eval_idx] = calibrator.predict_proba(xgb_validation_probabilities[eval_idx])
        assignments[eval_idx] = fold
        records.append({
            'calibration_fold': fold, 'fit_n': int(len(fit_idx)), 'eval_n': int(len(eval_idx)),
            'fit_unique_groups': int(len(fit_groups)), 'eval_unique_groups': int(len(eval_groups)),
            'group_overlap': int(len(overlap)),
            'raw_eval_ece': compute_ece(xgb_validation_probabilities[eval_idx], y_validation[eval_idx]),
            'calibrated_eval_ece': compute_ece(calibrated[eval_idx], y_validation[eval_idx]),
            'raw_eval_brier': compute_brier(xgb_validation_probabilities[eval_idx], y_validation[eval_idx]),
            'calibrated_eval_brier': compute_brier(calibrated[eval_idx], y_validation[eval_idx]),
        })

    if (assignments < 1).any() or not np.isfinite(calibrated).all():
        raise RuntimeError('Cross-fitted calibration tidak lengkap.')
    validate_probability_matrix(calibrated, 'cross-fitted calibrated validation')
    deployment_calibrator = build_platt(seed, 'platt_calibration_full_validation')
    deployment_calibrator.fit(xgb_validation_probabilities, y_validation)
    if not np.array_equal(deployment_calibrator.classes_, np.arange(N_CLASSES)):
        raise RuntimeError('Full-validation Platt tidak memiliki tiga kelas lengkap.')
    return calibrated, assignments, deployment_calibrator, records


### Build a seed-specific stacking and calibration run

Creates the semantic protocol manifest, trains stacking models on OOF meta-features, evaluates validation predictions, applies the complete validation calibrator to locked-test probabilities, and records private per-seed artifacts.


In [ ]:
def r1_07_protocol(seed_input: dict) -> dict:
    return {
        'protocol_version': PROTOCOL_VERSION,
        'seed': seed_input['seed'],
        'source_r1_06_protocol_hash': seed_input['protocol']['protocol_hash'],
        'source_r1_06_protocol_version': seed_input['protocol']['scientific_protocol']['protocol_version'],
        'meta_feature_names': META_FEATURE_NAMES,
        'xgboost_config': XGB_CONFIG,
        'calibration_config': CALIBRATION_CONFIG,
        'wa_weight_source': 'OOF macro-F1 only',
        'validation_calibrated_source': 'five-fold group-aware cross-fitted Platt',
        'locked_calibrated_source': 'Platt fit on full validation',
        'locked_test_policy': 'label-blind prediction only, no metrics',
        'model_order': MODEL_ORDER,
    }


def required_output_files(output_dir: Path) -> list[Path]:
    names = [
        'models/xgboost_12_feature.joblib',
        'models/calibrated_xgboost_platt_full_validation.joblib',
        'oof_model_predictions_fit_only.csv',
        'validation_model_predictions.csv',
        'locked_test_model_predictions_blind.csv',
        'validation_metrics.csv',
        'calibration_crossfit_validation_predictions.csv',
        'calibration_crossfit_summary.csv',
        'weighted_average_summary.json',
        'xgboost_feature_importance_gain.csv',
        'schema.json', 'run_manifest.json', 'run_state.json',
    ]
    return [output_dir / name for name in names]


def completed_output_is_valid(output_dir: Path, seed_input: dict, protocol_hash: str) -> bool:
    try:
        manifest = json.loads((output_dir / 'run_manifest.json').read_text(encoding='utf-8'))
        state = json.loads((output_dir / 'run_state.json').read_text(encoding='utf-8'))
        if state.get('status') != 'seed_completed' or manifest.get('seed') != seed_input['seed']:
            return False
        if manifest.get('protocol_hash') != protocol_hash or manifest.get('r1_06_input_hashes') != seed_input['input_hashes']:
            return False
        if manifest.get('locked_test_metrics_computed') is not False or manifest.get('locked_test_artifact_is_label_blind') is not True:
            return False
        if any(not path.is_file() for path in required_output_files(output_dir)):
            return False
        for relative_path, expected_hash in manifest['output_checksums'].items():
            if sha256_file(output_dir / relative_path) != expected_hash:
                return False
        locked = pd.read_csv(output_dir / 'locked_test_model_predictions_blind.csv')
        if 'y_silver' in locked or any('label' in column for column in locked.columns):
            return False
        for model_key, _ in MODEL_ORDER:
            extract_probabilities(locked, model_key, 'completed locked test')
        return True
    except Exception:
        return False


def process_seed(seed: int) -> dict:
    seed_input = load_r1_06_seed(seed)
    protocol = r1_07_protocol(seed_input)
    protocol_hash = canonical_json_hash(protocol)
    output_dir = seed_output_directory(seed)
    if completed_output_is_valid(output_dir, seed_input, protocol_hash):
        print(f'Stacking seed {seed}: completed output valid, skipped.')
        return {'seed': seed, 'status': 'skipped_completed', 'output_dir': str(output_dir)}

    output_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    atomic_write_json(output_dir / 'run_state.json', {
        'seed': seed, 'status': 'running', 'started_at_utc': utc_now(),
        'protocol_hash': protocol_hash, 'r1_06_input_hashes': seed_input['input_hashes'],
    })

    try:
        oof_base, validation_base, locked_base = seed_input['oof_base'], seed_input['validation_base'], seed_input['locked_base']
        oof_meta, validation_meta, locked_meta = seed_input['oof_meta'], seed_input['validation_meta'], seed_input['locked_meta']
        y_oof = oof_base['y_silver'].to_numpy(dtype=np.int64)
        y_validation = validation_base['y_silver'].to_numpy(dtype=np.int64)
        x_oof = oof_meta[META_FEATURE_NAMES].to_numpy(dtype=np.float64)
        x_validation = validation_meta[META_FEATURE_NAMES].to_numpy(dtype=np.float64)
        x_locked = locked_meta[META_FEATURE_NAMES].to_numpy(dtype=np.float64)

        svm_oof, bert_oof = extract_probabilities(oof_base, 'svm', 'OOF'), extract_probabilities(oof_base, 'bert', 'OOF')
        svm_validation, bert_validation = extract_probabilities(validation_base, 'svm', 'validation'), extract_probabilities(validation_base, 'bert', 'validation')
        svm_locked, bert_locked = extract_probabilities(locked_base, 'svm', 'locked test'), extract_probabilities(locked_base, 'bert', 'locked test')

        wa_weights = derive_wa_weights(y_oof, svm_oof, bert_oof)
        wa_oof = weighted_average(svm_oof, bert_oof, wa_weights)
        wa_validation = weighted_average(svm_validation, bert_validation, wa_weights)
        wa_locked = weighted_average(svm_locked, bert_locked, wa_weights)

        xgb_model = build_xgboost(seed)
        xgb_model.fit(x_oof, y_oof, sample_weight=compute_sample_weight(class_weight='balanced', y=y_oof))
        if not np.array_equal(xgb_model.classes_, np.arange(N_CLASSES)):
            raise RuntimeError('XGBoost tidak memiliki tiga kelas lengkap.')
        xgb_oof = xgb_model.predict_proba(x_oof)
        xgb_validation = xgb_model.predict_proba(x_validation)
        xgb_locked = xgb_model.predict_proba(x_locked)
        for name, probabilities in [('XGB OOF', xgb_oof), ('XGB validation', xgb_validation), ('XGB locked blind', xgb_locked)]:
            validate_probability_matrix(probabilities, name)

        calibrated_validation, calibration_fold, deployment_calibrator, calibration_records = fit_cross_fitted_platt(
            xgb_validation, y_validation, seed_input['validation_groups'], seed,
        )
        calibrated_locked = deployment_calibrator.predict_proba(xgb_locked)
        validate_probability_matrix(calibrated_locked, 'calibrated XGB locked blind')

        oof_predictions = model_prediction_frame(
            oof_base['row_id'].to_numpy(dtype=np.int64),
            {'svm': svm_oof, 'indobert': bert_oof, 'wa_svm_bert': wa_oof, 'xgboost': xgb_oof},
            y_oof,
            MODEL_ORDER[:-1],
        )
        validation_probabilities = {
            'svm': svm_validation, 'indobert': bert_validation, 'wa_svm_bert': wa_validation,
            'xgboost': xgb_validation, 'calibrated_xgboost': calibrated_validation,
        }
        locked_probabilities = {
            'svm': svm_locked, 'indobert': bert_locked, 'wa_svm_bert': wa_locked,
            'xgboost': xgb_locked, 'calibrated_xgboost': calibrated_locked,
        }
        validation_predictions = model_prediction_frame(
            validation_base['row_id'].to_numpy(dtype=np.int64), validation_probabilities, y_validation,
        )
        locked_predictions = model_prediction_frame(
            locked_base['row_id'].to_numpy(dtype=np.int64), locked_probabilities, None,
        )
        if 'y_silver' in locked_predictions:
            raise RuntimeError('Locked-test prediction tidak boleh memiliki y_silver.')

        validation_metrics = pd.DataFrame([{
            'seed': seed, 'split': 'validation', 'model_key': model_key, 'model': model_name,
            'probability_source': (
                'cross_fitted_platt_validation' if model_key == 'calibrated_xgboost'
                else 'five_fold_base_ensemble' if model_key in {'svm', 'indobert'}
                else 'oof_weighted_probability_fusion' if model_key == 'wa_svm_bert'
                else 'xgboost_fitted_on_oof_meta_features'
            ),
            **metric_summary(y_validation, validation_probabilities[model_key]),
        } for model_key, model_name in MODEL_ORDER])

        calibration_crossfit = pd.DataFrame({
            'row_id': validation_base['row_id'].to_numpy(dtype=np.int64),
            'y_silver': y_validation, 'calibration_fold': calibration_fold,
        })
        for index, label in enumerate(LABEL_NAMES):
            calibration_crossfit[f'xgboost_prob_{label}'] = xgb_validation[:, index]
            calibration_crossfit[f'calibrated_xgboost_prob_{label}'] = calibrated_validation[:, index]
        calibration_summary = pd.DataFrame(calibration_records)
        calibration_summary['seed'] = seed
        calibration_summary['method'] = CALIBRATION_CONFIG['method']
        calibration_summary['raw_minus_calibrated_ece'] = calibration_summary['raw_eval_ece'] - calibration_summary['calibrated_eval_ece']
        calibration_summary['raw_minus_calibrated_brier'] = calibration_summary['raw_eval_brier'] - calibration_summary['calibrated_eval_brier']

        atomic_dump_joblib(output_dir / 'models' / 'xgboost_12_feature.joblib', xgb_model)
        atomic_dump_joblib(output_dir / 'models' / 'calibrated_xgboost_platt_full_validation.joblib', deployment_calibrator)
        atomic_write_csv(output_dir / 'oof_model_predictions_fit_only.csv', oof_predictions)
        atomic_write_csv(output_dir / 'validation_model_predictions.csv', validation_predictions)
        atomic_write_csv(output_dir / 'locked_test_model_predictions_blind.csv', locked_predictions)
        atomic_write_csv(output_dir / 'validation_metrics.csv', validation_metrics)
        atomic_write_csv(output_dir / 'calibration_crossfit_validation_predictions.csv', calibration_crossfit)
        atomic_write_csv(output_dir / 'calibration_crossfit_summary.csv', calibration_summary)
        atomic_write_csv(output_dir / 'xgboost_feature_importance_gain.csv', feature_importance_frame(xgb_model))
        atomic_write_json(output_dir / 'weighted_average_summary.json', {
            'seed': seed, 'weight_source': wa_weights['source'],
            'weights': {'svm': wa_weights['svm'], 'indobert': wa_weights['indobert']},
            'oof_macro_f1': {'svm': wa_weights['svm_oof_macro_f1'], 'indobert': wa_weights['indobert_oof_macro_f1']},
        })
        atomic_write_json(output_dir / 'schema.json', {
            'meta_feature_names': META_FEATURE_NAMES,
            'model_order': [{'model_key': key, 'model': name} for key, name in MODEL_ORDER],
            'oof_note': 'XGBoost OOF output is fit-only provenance and is not an unbiased XGBoost metric.',
            'validation_calibrated_note': 'Calibrated XGBoost validation probabilities are five-fold group-aware cross-fitted.',
            'locked_test_calibrated_note': 'Calibrated XGBoost locked-test probabilities use Platt fit on all validation rows.',
        })

        output_names = [
            'models/xgboost_12_feature.joblib', 'models/calibrated_xgboost_platt_full_validation.joblib',
            'oof_model_predictions_fit_only.csv', 'validation_model_predictions.csv',
            'locked_test_model_predictions_blind.csv', 'validation_metrics.csv',
            'calibration_crossfit_validation_predictions.csv', 'calibration_crossfit_summary.csv',
            'weighted_average_summary.json', 'xgboost_feature_importance_gain.csv', 'schema.json',
        ]
        output_checksums = {name: sha256_file(output_dir / name) for name in output_names}
        completed_at = utc_now()
        atomic_write_json(output_dir / 'run_manifest.json', {
            'seed': seed, 'protocol': protocol, 'protocol_hash': protocol_hash,
            'r1_06_input_hashes': seed_input['input_hashes'],
            'r1_06_run_manifest_sha256': sha256_file(seed_input['input_dir'] / 'run_manifest.json'),
            'completed_at_utc': completed_at, 'locked_test_metrics_computed': False,
            'locked_test_artifact_is_label_blind': True, 'output_checksums': output_checksums,
            'environment': {
                'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__,
                'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'xgboost': xgboost.__version__,
            },
        })
        atomic_write_json(output_dir / 'run_state.json', {
            'seed': seed, 'status': 'seed_completed',
            'started_at_utc': datetime.fromtimestamp(started, timezone.utc).isoformat(),
            'completed_at_utc': completed_at, 'elapsed_seconds': time.time() - started,
            'protocol_hash': protocol_hash,
        })
        print(f'Stacking seed {seed} selesai dalam {time.time() - started:.1f} detik.')
        display(validation_metrics[['model', 'accuracy', 'macro_f1', 'ece', 'brier_score']])
        return {'seed': seed, 'status': 'completed', 'output_dir': str(output_dir)}
    except Exception as error:
        atomic_write_json(output_dir / 'run_state.json', {
            'seed': seed, 'status': 'failed', 'failed_at_utc': utc_now(),
            'protocol_hash': protocol_hash, 'error': repr(error),
        })
        raise
    finally:
        gc.collect()


### Execute or resume requested seeds

Runs the stacking and calibration workflow for each requested seed, skips completed compatible artifacts, and persists private models, predictions, metric tables, and checksums in the authorized environment.


In [ ]:
if not RUN_SEEDS or len(set(RUN_SEEDS)) != len(RUN_SEEDS):
    raise ValueError('RUN_SEEDS harus berisi seed unik.')
if not set(RUN_SEEDS).issubset(set(PLANNED_SEEDS)):
    raise ValueError(f'RUN_SEEDS harus subset dari {PLANNED_SEEDS}.')

run_results = []
new_count = 0
for current_seed in RUN_SEEDS:
    if MAX_NEW_SEEDS_PER_RUN is not None and new_count >= MAX_NEW_SEEDS_PER_RUN:
        print(f'Safe stop setelah {new_count} seed baru.')
        break
    result = process_seed(current_seed)
    run_results.append(result)
    if result['status'] == 'completed':
        new_count += 1

display(pd.DataFrame(run_results))


### Verify label-blind output integrity

Checks that final locked-test predictions remain label-blind, that five-model predictions and metrics are internally consistent, and that completed seed artifacts satisfy the stacking-and-calibration guardrails.


In [ ]:
def quality_check_r1_07(seed: int) -> dict:
    output_dir = seed_output_directory(seed)
    manifest = json.loads((output_dir / 'run_manifest.json').read_text(encoding='utf-8'))
    state = json.loads((output_dir / 'run_state.json').read_text(encoding='utf-8'))
    if state.get('status') != 'seed_completed':
        raise RuntimeError(f'Seed {seed} belum completed.')
    if manifest.get('locked_test_metrics_computed') is not False or manifest.get('locked_test_artifact_is_label_blind') is not True:
        raise RuntimeError('Stacking and calibration locked-test guardrail gagal.')
    failures = [
        relative_path for relative_path, expected_hash in manifest['output_checksums'].items()
        if sha256_file(output_dir / relative_path) != expected_hash
    ]
    validation = pd.read_csv(output_dir / 'validation_model_predictions.csv')
    locked = pd.read_csv(output_dir / 'locked_test_model_predictions_blind.csv')
    if 'y_silver' not in validation or 'y_silver' in locked or any('label' in column for column in locked.columns):
        raise RuntimeError('Skema label validation atau locked test tidak benar.')
    for frame, name in [(validation, 'validation'), (locked, 'locked test blind')]:
        for model_key, _ in MODEL_ORDER:
            extract_probabilities(frame, model_key, name)
    return {
        'seed': seed, 'status': state['status'], 'validation_rows': len(validation),
        'locked_test_blind_rows': len(locked), 'checksums_checked': len(manifest['output_checksums']),
        'checksum_failures': len(failures), 'locked_test_has_label': 'y_silver' in locked,
    }


quality_report = pd.DataFrame([
    quality_check_r1_07(seed) for seed in RUN_SEEDS
    if (seed_output_directory(seed) / 'run_manifest.json').is_file()
])
display(quality_report)
if not quality_report.empty and int(quality_report['checksum_failures'].sum()) == 0:
    print('Quality check passed. Stacking and calibration outputs are ready for repeated-seed aggregation after the remaining seeds are available.')
